Homework 3: Concurrent Echo Server Testing
This notebook builds the modified echo server, starts it, and verifies concurrent client behavior with the CLIENTS command and the client limit.

In [2]:
import json
import os
import pathlib
import socket
import subprocess
import threading
import time

root = pathlib.Path.cwd().resolve()
if not (root / 'Sockets' / 'echo-server').exists():
    root = root.parent

server_dir = root / 'Sockets' / 'echo-server'
binary = server_dir / 'build' / 'echo_server'

if not binary.exists():
    print('Building echo server...')
    subprocess.run(['cmake', '-S', str(server_dir), '-B', str(server_dir / 'build')], check=True, cwd=root)
    subprocess.run(['cmake', '--build', str(server_dir / 'build')], check=True, cwd=root)

print('Server binary:', binary)

server_proc = subprocess.Popen([
    str(binary)
], cwd=root, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait until the server is ready.
for _ in range(20):
    if server_proc.poll() is not None:
        raise RuntimeError('Server exited unexpectedly before starting.')
    time.sleep(0.2)

print('Server started with PID', server_proc.pid)

HOST = '127.0.0.1'
PORT = 9090

# Helper to open a client socket and return it.
def open_client():
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.connect((HOST, PORT))
    return sock

# Read a single line reply from the server.
def read_line(sock, timeout=5.0):
    sock.settimeout(timeout)
    data = b''
    while True:
        chunk = sock.recv(1024)
        if not chunk:
            break
        data += chunk
        if b'\n' in data:
            break
    return data.decode('utf-8', errors='replace')

clients = []

try:
    print('Opening 3 concurrent clients...')
    for i in range(3):
        sock = open_client()
        clients.append(sock)
        print(f'Client {i+1} connected')

    print('\nAsking each client for CLIENTS count:')
    for i, sock in enumerate(clients, start=1):
        sock.sendall(b'CLIENTS\n')
        response = read_line(sock)
        print(f'Client {i} response:', repr(response))

    print('\nOpening 4th client to verify rejection...')
    sock4 = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock4.settimeout(5.0)
    try:
        sock4.connect((HOST, PORT))
        response4 = read_line(sock4)
        print('Fourth client response:', repr(response4))
    except Exception as exc:
        print('Fourth client failed to connect or read:', exc)
    finally:
        sock4.close()

    print('\nClosing one client and then opening a replacement...')
    clients[0].sendall(b'QUIT\n')
    response = read_line(clients[0])
    print('Client 1 quit response:', repr(response))
    clients[0].close()

    time.sleep(0.5)

    print('Opening new client after disconnect...')
    replacement = open_client()
    replacement.sendall(b'CLIENTS\n')
    response = read_line(replacement)
    print('Replacement client response:', repr(response))
    replacement.sendall(b'QUIT\n')
    print('Replacement QUIT response:', repr(read_line(replacement)))
    replacement.close()

    print('\nVerifying remaining clients see updated count:')
    for i, sock in enumerate(clients[1:], start=2):
        sock.sendall(b'CLIENTS\n')
        response = read_line(sock)
        print(f'Client {i} response after replacement:', repr(response))

    for sock in clients[1:]:
        sock.sendall(b'QUIT\n')
        print('Remaining client quit response:', repr(read_line(sock)))
        sock.close()

finally:
    if server_proc.poll() is None:
        server_proc.terminate()
        try:
            server_proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server_proc.kill()
            server_proc.wait()

print('Server terminated.')

Server binary: /workspaces/cecs327/Sockets/echo-server/build/echo_server
Server started with PID 33437
Opening 3 concurrent clients...
Client 1 connected
Client 2 connected
Client 3 connected

Asking each client for CLIENTS count:
Client 1 response: 'CLIENTS 1\n'
Client 2 response: 'CLIENTS 3\n'
Client 3 response: 'CLIENTS 3\n'

Opening 4th client to verify rejection...
Fourth client response: 'ERROR too many clients\n'

Closing one client and then opening a replacement...
Client 1 quit response: 'GOODBYE\n'
Opening new client after disconnect...
Replacement client response: 'CLIENTS 3\n'
Replacement QUIT response: 'GOODBYE\n'

Verifying remaining clients see updated count:
Client 2 response after replacement: 'CLIENTS 2\n'
Client 3 response after replacement: 'CLIENTS 2\n'
Remaining client quit response: 'GOODBYE\n'
Remaining client quit response: 'GOODBYE\n'
Server terminated.
